# AWS AI/ML Track - Week 1: Small Python Data-Processing App

This notebook demonstrates an end-to-end data processing workflow:
1. **Load Raw Dataset**: Imports a raw CSV dataset (or creates a sample in-memory fallback).
2. **Data Cleaning**: Uses modular Python functions to handle duplicates, currency strings, outliers, and missing values.
3. **NumPy Processing**: Converts data into NumPy arrays, calculates statistics (`mean`, `std`, `percentiles`), and applies vectorized Z-score normalization.

> **Compatibility**: This notebook runs seamlessly in both **Local VS Code Jupyter** and **Google Colab**.

## 1. Setup & Imports

In [ ]:
import os
import pandas as pd
import numpy as np

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version:  {np.__version__}")

## 2. Load Raw Dataset
We check if `raw_dataset.csv` exists in the current folder. If running on Google Colab without uploading files, this cell automatically creates the dataset so you can run all cells immediately.

In [ ]:
CSV_FILENAME = "raw_dataset.csv"

# Ensure the dataset exists (ideal for Google Colab)
if not os.path.exists(CSV_FILENAME):
    sample_data = """customer_id,service_type,usage_hours,monthly_spend,latency_ms
C101,Cloud_Compute,120.5,$245.80,32.4
C102,Cloud_Storage,45.0,$45.00,18.2
C103,AI_Inference,,$310.50,110.6
C104,Cloud_Compute,300.2,$590.00,
C105,Database,80.0,$120.00,24.1
C106,AI_Inference,-15.0,$180.20,95.0
C107,Cloud_Storage,60.5,$60.50,19.8
C101,Cloud_Compute,120.5,$245.80,32.4
C108,Database,150.0,$210.00,28.7
C109,AI_Inference,450.0,$920.40,140.2
C110,Cloud_Compute,220.0,,42.1
C111,Cloud_Storage,15.0,$20.00,15.5
C112,AI_Inference,310.0,$640.00,115.0
C113,Database,5000.0,$250.00,31.0
C114,Cloud_Compute,95.0,$190.00,35.8
C115,AI_Inference,180.5,$370.10,88.4
"""
    with open(CSV_FILENAME, "w") as f:
        f.write(sample_data.strip())
    print(f"[INFO] Generated sample '{CSV_FILENAME}' for this session.")

raw_df = pd.read_csv(CSV_FILENAME)
print(f"Raw dataset loaded with shape: {raw_df.shape}")
raw_df

## 3. Data Cleaning with Functions

Notice the issues in the raw data:
- **Duplicates**: `C101` appears twice.
- **Currency formatting**: `monthly_spend` contains `$` symbols.
- **Missing values**: `C103` has missing `usage_hours`, `C104` has missing `latency_ms`.
- **Outliers / Invalid values**: `C106` has negative usage (`-15.0`), and `C113` has unrealistic usage (`5000.0` hrs).

In [ ]:
def clean_currency(series: pd.Series) -> pd.Series:
    """Strips '$' and commas, returning clean float values."""
    return (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"None": np.nan, "nan": np.nan, "": np.nan})
        .astype(float)
    )

def clean_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cleans dataset using structured steps:
    1. Deduplicates records
    2. Converts formats & numeric types
    3. Replaces invalid values/outliers with NaN
    4. Imputes missing values with column medians
    """
    data = df.copy()
    
    # 1. Deduplicate
    data = data.drop_duplicates(subset=["customer_id"])
    
    # 2. Format columns
    data["monthly_spend"] = clean_currency(data["monthly_spend"])
    data["usage_hours"] = pd.to_numeric(data["usage_hours"], errors="coerce")
    data["latency_ms"] = pd.to_numeric(data["latency_ms"], errors="coerce")
    
    # 3. Filter invalid / outlier values (set to NaN)
    data.loc[data["usage_hours"] <= 0, "usage_hours"] = np.nan
    data.loc[data["usage_hours"] > 2000, "usage_hours"] = np.nan
    
    # 4. Impute missing values with column median
    for col in ["usage_hours", "monthly_spend", "latency_ms"]:
        col_median = data[col].median()
        data[col] = data[col].fillna(col_median)
        
    return data.reset_index(drop=True)

cleaned_df = clean_dataset(raw_df)
print(f"Cleaned dataset: {len(cleaned_df)} rows (0 missing values)")
cleaned_df

## 4. Processing with NumPy

We now leverage NumPy for numerical computation:
- Extract a 2D matrix of features (`usage_hours`, `monthly_spend`, `latency_ms`).
- Perform vectorized element-wise math (hourly spend rate).
- Calculate statistical measures using NumPy functions.
- Apply Z-Score normalization: $$Z = \frac{X - \mu}{\sigma}$$

In [ ]:
# 1. Extract 2D NumPy array
features = cleaned_df[["usage_hours", "monthly_spend", "latency_ms"]].to_numpy()
print("NumPy Feature Matrix Shape:", features.shape)
print("Data type:", features.dtype)

# 2. Vectorized computation: Hourly Cost Rate ($ / hour)
usage_vector = features[:, 0]
spend_vector = features[:, 1]
hourly_rates = np.round(spend_vector / usage_vector, 2)
print("\nVectorized Hourly Rates (First 5 records):")
print(hourly_rates[:5])

# 3. Statistical Calculations with NumPy
feature_names = ["Usage Hours", "Monthly Spend ($)", "Latency (ms)"]
means = np.mean(features, axis=0)
stds = np.std(features, axis=0)
mins = np.min(features, axis=0)
maxs = np.max(features, axis=0)
medians = np.median(features, axis=0)

stats_df = pd.DataFrame({
    "Feature": feature_names,
    "Mean": np.round(means, 2),
    "Std Dev": np.round(stds, 2),
    "Median": np.round(medians, 2),
    "Min": np.round(mins, 2),
    "Max": np.round(maxs, 2)
})
print("\n--- NumPy Computed Summary Statistics ---")
display(stats_df)

# 4. Vectorized Z-Score Normalization
normalized_features = (features - means) / stds
print("\nNormalized Features (First 3 samples):")
print(np.round(normalized_features[:3], 3))

## 5. Verification & Summary Output

In [ ]:
print("=== Task Verification Checklist ===")
print(f"✔ Raw dataset loaded: {len(raw_df)} rows")
print(f"✔ Cleaned dataset processed: {len(cleaned_df)} rows")
print(f"✔ NumPy operations completed: shape {features.shape}")
print(f"✔ Mean of normalized features (should be ~0): {np.round(np.mean(normalized_features, axis=0), 4)}")
print(f"✔ Std of normalized features (should be 1):    {np.round(np.std(normalized_features, axis=0), 4)}")
print("\nPipeline execution successful!")